In [2]:
#Phase 1: Environment Setup & Data Preparation
# Cell 1: Install required libraries
!pip install -q transformers datasets accelerate scikit-learn fastapi uvicorn nest-asyncio

# Cell 2: Import libraries and load data
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Load the dataset (Make sure you upload 'customer_support_tickets.csv' to Colab files)
df = pd.read_csv('customer_support_tickets.csv')

# Drop missing values in crucial columns
df = df.dropna(subset=['Ticket Description', 'Ticket Type'])

# Create label mappings
labels = df['Ticket Type'].unique().tolist()
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}

df['label'] = df['Ticket Type'].map(label2id)
df = df.rename(columns={'Ticket Description': 'text'})

# Keep only the columns we need for NLP
df_clean = df[['text', 'label']].copy()

# Split into train and test sets (80/20 split)
train_df, test_df = train_test_split(df_clean, test_size=0.2, random_state=42, stratify=df_clean['label'])

print(f"Total categories: {len(labels)}")
print(f"Training samples: {len(train_df)} | Test samples: {len(test_df)}")

Total categories: 5
Training samples: 6775 | Test samples: 1694


In [3]:
#Phase 2: tokenization
# Cell 3: Tokenization
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    # Truncate and pad to max length of 256 tokens to speed up training in Colab
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

# Convert pandas dataframes to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Map tokenization across datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6775 [00:00<?, ? examples/s]

Map:   0%|          | 0/1694 [00:00<?, ? examples/s]

In [5]:
#Phase 3: Model Fine-Tuning
# Cell 4: Metrics evaluation definition
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Cell 5: Initialize Model and Training Arguments
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",        # Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)
# Cell 6: Run Training
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.611766,1.610505,0.204841,0.069197,0.061093,0.197763
2,1.611475,1.612497,0.207202,0.071807,0.161306,0.201136
3,1.605583,1.610654,0.197757,0.132615,0.174464,0.192642


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1272, training_loss=1.6094010341092475, metrics={'train_runtime': 173.4721, 'train_samples_per_second': 117.166, 'train_steps_per_second': 7.333, 'total_flos': 1346271961536000.0, 'train_loss': 1.6094010341092475, 'epoch': 3.0})

In [6]:
#Phase 4: Evaluation
# Cell 7: Evaluate Model
eval_results = trainer.evaluate()
print("\n--- FINAL METRICS FOR YOUR RESUME ---")
print(f"Accuracy: {eval_results['eval_accuracy'] * 100:.2f}%")
print(f"Macro F1-Score: {eval_results['eval_f1']:.4f}")
print(f"Test Set Size: {len(test_df)}")

# Save the model and tokenizer locally
model.save_pretrained("./best_model")
tokenizer.save_pretrained("./best_model")

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
1.605583,1.610654,3,0.197757,0.132615,0.174464,0.192642



--- FINAL METRICS FOR YOUR RESUME ---
Accuracy: 19.78%
Macro F1-Score: 0.1326
Test Set Size: 1694


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./best_model/tokenizer_config.json', './best_model/tokenizer.json')

In [7]:
# Cell 8: Quick Inference Test to verify it's working fine
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the saved model and tokenizer
MODEL_PATH = "./best_model"
test_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
test_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# Put model in evaluation mode
test_model.eval()

# Test string (You can change this text to test different categories!)
sample_ticket = "I am trying to log into my account but it says my password expired. I need to reset it ASAP."

# Tokenize the sample text
inputs = test_tokenizer(sample_ticket, return_tensors="pt", truncation=True, padding=True, max_length=256)

# Get prediction
with torch.no_grad():
    outputs = test_model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class_idx = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_class_idx].item()

# Output the results
predicted_label = test_model.config.id2label[predicted_class_idx]
print("=" * 50)
print(f"Input Ticket: '{sample_ticket}'")
print("-" * 50)
print(f"Predicted Category: {predicted_label}")
print(f"Confidence Score:   {confidence * 100:.2f}%")
print("=" * 50)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Input Ticket: 'I am trying to log into my account but it says my password expired. I need to reset it ASAP.'
--------------------------------------------------
Predicted Category: Technical issue
Confidence Score:   21.50%


In [9]:
#for ui
!pip install -q gradio

In [11]:
#converting to zip
!zip -r best_model.zip best_model

  adding: best_model/ (stored 0%)
  adding: best_model/tokenizer.json (deflated 71%)
  adding: best_model/tokenizer_config.json (deflated 43%)
  adding: best_model/config.json (deflated 54%)
  adding: best_model/model.safetensors (deflated 8%)


In [14]:
import time
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==========================================
# 1. LOAD FINE-TUNED MODEL & TOKENIZER
# ==========================================
MODEL_PATH = "./best_model"
ui_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
ui_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
ui_model.eval()

# ==========================================
# 2. DEFINE THE ENHANCED PREDICTION FUNCTION
# ==========================================
def classify_ticket_enhanced(ticket_text):
    if not ticket_text.strip():
        return (
            {},
            "⚠️ Please enter a ticket description",
            "0.0%",
            "0.0 ms"
        )

    start_time = time.time()

    # Tokenize input (max 256 tokens as per model constraints)
    inputs = ui_tokenizer(
        ticket_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # Run model inference
    with torch.no_grad():
        outputs = ui_model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]

    # Calculate inference latency
    latency_ms = (time.time() - start_time) * 1000

    # Build dictionary of all class probabilities for the visualization plot
    label_probs = {}
    for idx, prob in enumerate(probabilities):
        label = ui_model.config.id2label[idx]
        label_probs[label] = float(prob.item())

    # Get the top predicted class and confidence
    top_class_idx = torch.argmax(probabilities).item()
    top_label = ui_model.config.id2label[top_class_idx]
    top_confidence = probabilities[top_class_idx].item() * 100

    # Format outputs for the dashboard metrics
    formatted_category = f"🎯 {top_label}"
    formatted_confidence = f"{top_confidence:.1f}%"
    formatted_latency = f"{latency_ms:.1f} ms"

    return label_probs, formatted_category, formatted_confidence, formatted_latency


# ==========================================
# 3. BUILD THE GRADIO DASHBOARD (UI)
# ==========================================
custom_css = """
.container { max-width: 1100px; margin: 0 auto; padding-top: 20px; }
.header { text-align: center; margin-bottom: 25px; padding: 20px; border-radius: 12px; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"), css=custom_css) as demo:

    # Header Banner
    with gr.Row(elem_classes="header"):
        with gr.Column():
            gr.Markdown(
                """
                # 🛠️ AI Customer Support Ticket Classifier
                ### Powered by Fine-Tuned DistilBERT | Instant Multi-Class Classification
                """
            )

    # Main Dashboard Body
    with gr.Row():

        # LEFT COLUMN: User Input Area
        with gr.Column(scale=1):
            gr.Markdown("### **1. Submit Ticket Description**")
            gr.Markdown("ℹ️ *Model processes up to 256 tokens dynamically.*")

            input_text = gr.Textbox(
                lines=6,
                placeholder="Enter the full customer support ticket text here... (e.g., product setup issues, billing questions, cancellation requests)",
                label="Ticket Description",
                show_label=False
            )

            submit_btn = gr.Button("🚀 Classify Ticket", variant="primary")

            # Interactive Examples Helper
            gr.Examples(
                examples=[
                    ["My GoPro Hero screen is completely cracked and the device won't boot up. I tried following the troubleshooting manual but the issue persists."],
                    ["I received an accidental charge of $49.99 on my credit card this morning for a subscription I canceled last week. I need a chargeback or refund immediately."],
                    ["How do I sync my new LG Smart TV with my peripheral Bluetooth speakers? The user instructions don't mention compatibility setup steps."]
                ],
                inputs=input_text,
                label="Quick-Load Sample Tickets"
            )

        # RIGHT COLUMN: Model Insights & Results
        with gr.Column(scale=1):
            gr.Markdown("### **2. AI Prediction & Insights**")

            # Probabilities distribution chart
            output_chart = gr.Label(num_top_classes=5, label="Category Probability Distribution")

            # Native UI input boxes for metrics to prevent white-on-white text rendering issues
            metric_category = gr.Textbox(
                label="Final Predicted Category",
                value="Awaiting input...",
                interactive=False
            )

            with gr.Row():
                metric_confidence = gr.Textbox(
                    label="Model Confidence",
                    value="-- %",
                    interactive=False
                )
                metric_latency = gr.Textbox(
                    label="Inference Latency",
                    value="-- ms",
                    interactive=False
                )

            # Explanatory Info Card
            with gr.Accordion("About the Model", open=False):
                gr.Markdown(
                    """
                    * **Base Architecture:** `distilbert-base-uncased`
                    * **Classes Evaluated:** 5 Support Categories (*Refund request, Technical issue, Cancellation request, Product inquiry, Billing inquiry*)
                    * **Under the Hood:** Tokenizes sequence strings, runs a forward pass on PyTorch weights, and calculates softmax confidence scores dynamically.
                    """
                )

    # Button Event Handlers
    submit_btn.click(
        fn=classify_ticket_enhanced,
        inputs=input_text,
        outputs=[output_chart, metric_category, metric_confidence, metric_latency]
    )

    # Real-time update as you press enter in the textbox
    input_text.submit(
        fn=classify_ticket_enhanced,
        inputs=input_text,
        outputs=[output_chart, metric_category, metric_confidence, metric_latency]
    )

# Launch the UI
demo.launch(inline=True, share=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_447/3058787009.py:72: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"), css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c93d61a3a797f84885.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
